In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30

Autosaving every 30 seconds


# Predict with the Flow Matching Model or CGM

In [ ]:
import importlib
import pickle

import hydra
import lightning as L
import numpy as np
import pandas as pd
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import FORECAST_ENS_FLAT_AGG_PATH, OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import (
    compute_scores_per_leadtime,
    log_scores,
    save_predictions_dataarray,
    save_scores_df,
    update_wandb_run,
)
from genpp.models.scores import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

torch.set_float32_matmul_precision("high")

/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened bec

## Best Models
FM model:
- [x] `feik/genpp/pwb8kh5a` (std)
- [x] `feik/genpp/vg3d7e9s` (abs)

FM model UViT
- [x] `feik/genpp/ftmjxjq9` (std)
- [x] `feik/genpp/s19rsj2i` (abs)

Chen model:
-  [x] `feik/genpp/2f1vpjz0` (Full energy score loss)
-  [x] `feik/genpp/23phjuuc` (Patchwise Chen (3))
-  [x] `feik/genpp/ynl8hbdr` (Patchwise Chen (5))
-  [x] `feik/genpp/eu94vgqa` (Patchwise Chen (7))

Engression:
-  [x] `feik/genpp/hbuy7eio` (Full energy score loss)

In [3]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/n9j3cu0z"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )

In [5]:
# Do not shuffle any dataloader
cfg.data.module.dataloader_config.train.shuffle = False
cfg.data.module.dataloader_config.val.shuffle = False
cfg.data.module.dataloader_config.val.batch_size = 16  # For faster predictions
cfg.data.module.dataloader_config.test.shuffle = False


datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")
datamodule.setup(stage="test")

Configuration hash: 1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c.pt.


In [9]:
class_path = cfg.model._target_
module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)
try:
    model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)
except pickle.UnpicklingError:
    model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT, weights_only=False)

/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'backbone' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['backbone'])`.


In [ ]:
trainer = L.Trainer(logger=False, accelerator="gpu", devices=[1])

In [ ]:
test_scores = trainer.test(model, datamodule.test_dataloader())

In [ ]:
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

In [ ]:
# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

In [ ]:
x_ds = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

time_slice = {
    "train": hydra.utils.instantiate(cfg.data.splits.train),
    "val": hydra.utils.instantiate(cfg.data.splits.val),
    "test": hydra.utils.instantiate(cfg.data.splits.test),
}
val_slice = time_slice["val"]

x_ds = x_ds.stack(sample=("time", "prediction_timedelta"))


datamodule.cache_metadata

# Now lets get the actual times we need for the val set and and get the val data
init_times = datamodule.cache_metadata["feature_metadata"]["time"]["val"]
timedeltas = datamodule.cache_metadata["feature_metadata"]["prediction_timedelta"]["val"]
val_times = init_times + timedeltas
val_prediction = pd.MultiIndex.from_arrays(
    [init_times, timedeltas], names=["time", "prediction_timedelta"]
)

# Now load the y_val data for the val times
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=val_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", val_prediction),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_val = y_val.sel(feature=feature_order)
y_t = torch.from_numpy(y_val.values).to(predictions_rescaled)

In [ ]:
SKIP_VARIOGRAM = False  # Option to skip variogram score computation due to long computation times

# Compute the scores
crps_ens = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

crps_per_margin = crps_ens(predictions_rescaled, y_t)

# predictions_rescaled: [t, n, d, lat, lon] -> [b, n_samples, c, h, w]
# y_t: [t, d, lat, lon] -> [b, c, h, w]
# The loss functions now handle reshaping internally with mode parameter

# _u is for un-reduced scores
energy_score_per_var_u = es(predictions_rescaled, y_t, mode="per_var")
energy_score_full_u = es(predictions_rescaled, y_t, mode="complete")

# For the VS there are huge intermediary results, thats why it is computed batchwise
if not SKIP_VARIOGRAM:
    variogram_score_per_var_u = vs(predictions_rescaled, y_t, mode="per_var")
    variogram_score_full_u = vs(predictions_rescaled, y_t, mode="complete")

In [ ]:
# Reduce the scores
crps_per_var = reduce(crps_per_margin, "t d h w -> d", reduction="mean")
crps_full = reduce(crps_per_margin, "t d h w -> 1", "mean")
energy_score_per_var = reduce(energy_score_per_var_u, "t d -> d", "mean")
energy_score_full = reduce(energy_score_full_u, "t -> 1", "mean")
if not SKIP_VARIOGRAM:
    variogram_score_per_var = reduce(variogram_score_per_var_u, "t d -> d", "mean")
    variogram_score_full = reduce(variogram_score_full_u, "t -> 1", "mean")

In [ ]:
# Log the Scores
model_class = cfg.model._target_.split(".")[-1]


log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="CRPS",
    variables=datamodule.y_select_variables,
    scores=crps_per_var,
)
log_scores(
    file=SCORE_FILE, model=model_class, metric="CRPS", variables=["combined"], scores=crps_full
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=energy_score_per_var,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=["combined"],
    scores=energy_score_full,
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=variogram_score_per_var,
    )
    # Log full scores
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=["combined"],
        scores=variogram_score_full,
    )

In [ ]:
scores = compute_scores_per_leadtime(
    timedeltas,
    crps_per_margin,
    energy_score_per_var_u,
    energy_score_full_u,
    variogram_score_per_var_u if not SKIP_VARIOGRAM else None,
    variogram_score_full_u if not SKIP_VARIOGRAM else None,
    method=None,
)
scores

In [ ]:
# Update WandB run
full_scores = {"val": scores}
update_wandb_run(f"feik/genpp/{MODEL_ID}", full_scores)

# Also log as a table
records = []
for dataset, metrics in full_scores.items():
    for metric_name, horizons in metrics.items():
        for horizon, value in horizons.items():
            records.append((f"{model.__class__.__name__}", dataset, metric_name, horizon, value))
df = pd.DataFrame(records, columns=["method", "dataset", "metric", "horizon", "value"])

save_scores_df(df=df, run_path=f"feik/genpp/{MODEL_ID}")

In [ ]:
# Save the predictions as a DataArray
if hasattr(model, "n_samples_train"):
    N = np.arange(model.n_samples_train)
elif hasattr(model, "n_samples"):
    N = np.arange(model.n_samples)
else:
    raise ValueError("Model has no attribute n_samples or n_samples_train")

res = xr.DataArray(
    predictions_rescaled.cpu().numpy(),
    coords={
        "prediction": y_val.prediction,
        "sample": N,
        "feature": y_val.feature,
        "longitude": y_val.longitude,
        "latitude": y_val.latitude,
    },
    dims=("prediction", "sample", "feature", "longitude", "latitude"),
)
save_predictions_dataarray(
    predictions=res, save_path=MODEL_DIR / "val_predictions.zarr", overwrite=True
)